# Master Data Cleaning & Preparation

## Objective
This notebook implements the integrated cleaning logic from all 4 team members on the raw integrated dataset from Phase 2.
1. **Identity & Price (Lyna)**
2. **Storage & Memory (Leena)**
3. **Display Metrics (Aya)**
4. **Processing Power (Abdallah & Mimoun)**


In [ ]:
import pandas as pd
import numpy as np
import re
import warnings
warnings.filterwarnings('ignore')

print("Libraries imported successfully")

## 1. Load Integrated Raw Data
We load the dataset created in Phase 02.

In [ ]:
RAW_DATA_PATH = '../02_Data_Collection_Integration/integrated_raw_dataset.csv'
df = pd.read_csv(RAW_DATA_PATH)
print(f"Loaded Raw Dataset: {df.shape[0]} rows")

## 2. Part 1: Identity Cleaning (Lyna)
Standardizing Brands, Models, Conditions and Price outliers.

In [ ]:
# Normalize text
for col in ['LAPTOP_BRAND', 'LAPTOP_MODEL', 'LAPTOP_CONDITION', 'CITY']:
    df[col] = df[col].astype(str).str.upper().str.strip()

# 2.1 Brand Standardization & Inference
brand_mapping = {'MAC': 'APPLE', 'MACBOOK': 'APPLE', 'IMAC': 'APPLE'}
df['LAPTOP_BRAND'] = df['LAPTOP_BRAND'].replace(brand_mapping)

def infer_brand(row):
    if row['LAPTOP_BRAND'] not in ['NEEDTOBEFILLED', 'UNKNOWN', 'NAN']:
        return row['LAPTOP_BRAND']
    model = str(row['LAPTOP_MODEL']).upper()
    if any(x in model for x in ['LATITUDE', 'XPS', 'INSPIRON', 'PRECISION', 'ALIENWARE']): return 'DELL'
    if any(x in model for x in ['PAVILION', 'ELITEBOOK', 'OMEN', 'ENVY', 'PROBOOK', 'ZBOOK', 'VICTUS']): return 'HP'
    if any(x in model for x in ['THINKPAD', 'IDEAPAD', 'LEGION', 'YOGA']): return 'LENOVO'
    if any(x in model for x in ['VIVOBOOK', 'ZENBOOK', 'ROG', 'TUF']): return 'ASUS'
    if 'MACBOOK' in model: return 'APPLE'
    if 'SURFACE' in model: return 'MICROSOFT'
    return row['LAPTOP_BRAND']

df['LAPTOP_BRAND'] = df.apply(infer_brand, axis=1)

# 2.2 Condition Mapping
cond_map = {
    'BON TAT': 'Used - Good', 'BON ÉTAT': 'Used - Good', 'ETAT NEUF': 'Used - Excellent',
    'JAMAIS UTILIS': 'Never Used', 'NEUF JAMAIS UTILISÉ': 'Never Used', 'NEVER USED (NEW)': 'Never Used',
    'GOOD CONDITION': 'Used - Good', 'AVERAGE CONDITION': 'Used - Fair', 'MOYEN': 'Used - Fair'
}
df['LAPTOP_CONDITION'] = df['LAPTOP_CONDITION'].replace(cond_map)
df['LAPTOP_CONDITION'] = df['LAPTOP_CONDITION'].replace('NEEDTOBEFILLED', 'Used - Good') # Default

# 2.3 Price Conversion & Basic Filtering
df['PRICE'] = pd.to_numeric(df['PRICE'], errors='coerce')
df = df[df['PRICE'] > 5000] # Drop trash prices
print("Identity cleaning completed.")

## 3. Part 2: Storage & Memory Engineering (Leena)
Converting strings to GB and intelligent imputation.

In [ ]:
def parse_to_gb(s):
    if pd.isna(s) or str(s).upper() == 'NEEDTOBEFILLED': return 0.0
    s = str(s).upper()
    match = re.search(r'(\d+)', s)
    if not match: return 0.0
    val = float(match.group(1))
    if 'TB' in s or 'TO' in s: return val * 1024
    return val

df['RAM_GB'] = df['RAM_SIZE'].apply(parse_to_gb)
df['SSD_GB'] = df['SSD_SIZE'].apply(parse_to_gb)
df['HDD_GB'] = df['HDD_SIZE'].apply(parse_to_gb)

# If SSD/HDD are 0, try to parse from STORAGE_SIZE
mask_no_storage = (df['SSD_GB'] == 0) & (df['HDD_GB'] == 0)
df.loc[mask_no_storage, 'SSD_GB'] = df.loc[mask_no_storage, 'STORAGE_SIZE'].apply(parse_to_gb)

# Standardize RAM types
def clean_ram_type(s):
    s = str(s).upper()
    if 'DDR5' in s: return 'DDR5'
    if 'DDR4' in s: return 'DDR4'
    if 'DDR3' in s: return 'DDR3'
    return 'DDR4' # Modern standard

df['RAM_TYPE'] = df['RAM_TYPE'].apply(clean_ram_type)
print("Storage engineering completed.")

## 4. Part 3: Display Metrics (Aya)
Ensuring resolutions and sizes are valid.

In [ ]:
def clean_screen(s):
    if pd.isna(s) or s == 'NEEDTOBEFILLED': return 15.6
    match = re.search(r'(\d+\.?\d*)', str(s))
    if match:
        val = float(match.group(1))
        if 10 < val < 20: return val
    return 15.6

df['SCREEN_SIZE'] = df['SCREEN_SIZE'].apply(clean_screen)

def clean_res(s):
    s = str(s).upper()
    if 'FHD' in s or '1080' in s: return '1920x1080'
    if '4K' in s or '3840' in s: return '3840x2160'
    if 'QHD' in s or '1440' in s: return '2560x1440'
    return '1920x1080'

df['SCREEN_RESOLUTION'] = df['SCREEN_RESOLUTION'].apply(clean_res)
print("Screen cleaning completed.")

## 5. Part 4: Processing Power (Abdallah & Mimoun)
CPU and GPU Binning.

In [ ]:
def group_cpu(s):
    s = str(s).upper()
    if 'I7' in s: return 'Intel Core i7'
    if 'I5' in s: return 'Intel Core i5'
    if 'I3' in s: return 'Intel Core i3'
    if 'I9' in s: return 'Intel Core i9'
    if 'RYZEN 7' in s: return 'AMD Ryzen 7'
    if 'RYZEN 5' in s: return 'AMD Ryzen 5'
    if 'M1' in s or 'M2' in s or 'M3' in s or 'M4' in s: return 'Apple Silicon'
    return 'Other'

df['CPU_BRAND'] = df['CPU'].apply(group_cpu)

def group_gpu(s):
    s = str(s).upper()
    if 'RTX' in s: return 'Nvidia RTX'
    if 'GTX' in s: return 'Nvidia GTX'
    if 'RADEON' in s: return 'AMD Radeon'
    if 'APPLE' in s: return 'Apple GPU'
    return 'Integrated'

df['GPU_TYPE'] = df['DEDICATED_GPU'].apply(group_gpu)
print("CPU/GPU binning completed.")

## 6. Final Validation & Outlier Removal
Removing unrealistic entries.

In [ ]:
# Drop ultra-outlier prices (> 800k DZD)
df = df[df['PRICE'] < 800000]

# Final Clean Column Selection
final_df = df[[
    'PRICE', 'LAPTOP_BRAND', 'LAPTOP_MODEL', 'LAPTOP_CONDITION', 
    'CPU_BRAND', 'GPU_TYPE', 'RAM_GB', 'SSD_GB', 'HDD_GB',
    'SCREEN_SIZE', 'SCREEN_RESOLUTION', 'CITY', 'POST_YEAR', 'POST_MONTH'
]]

print(f"Final Clean Dataset Shape: {final_df.shape}")

## 7. Exporting Cleaned Dataset

In [ ]:
OUTPUT_FILE = 'cleaned_dataset.csv'
final_df.to_csv(OUTPUT_FILE, index=False)
print(f"Successfully exported cleaned dataset to: {OUTPUT_FILE}")